# 第 03 课 · 交错网格、离散梯度与压力投影法

> **学习模式：**独立编写 JAX CFD 求解器 | VS Code / Jupyter | 从公式到可运行代码

**本课目标**：写出相容的离散散度、梯度、拉普拉斯算子，并使用 FFT 求解压力泊松方程。

**前置知识**：二维扩散与对流差分，基础 NumPy / JAX。

**操作约定**：先逐单元执行，再尝试“实验与练习”；若改变了函数定义或网格设置，请重启内核并执行全部单元格。JAX 第一次调用包含编译时间。

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax import lax
jax.config.update("jax_enable_x64", True)
print("JAX:", jax.__version__, "devices:", jax.devices())

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


JAX: 0.12.0.1 devices: [CpuDevice(id=0)]


## 1. 为什么不可压缩流动需要压力？

$$
\nabla\cdot\mathbf u=0,\qquad
\frac{\partial\mathbf u}{\partial t}+\mathbf u\cdot\nabla\mathbf u
=-\frac{1}{\rho}\nabla p+\nu\nabla^2\mathbf u.
$$

压力在不可压缩问题中承担**约束速度场满足零散度**的作用。先得到没有施加压力的预测速度 $\mathbf u^*$，再通过压力增量修正：

$$
\mathbf u^{n+1}=\mathbf u^*-\Delta t\nabla\phi,
\qquad \nabla^2\phi=\frac{1}{\Delta t}\nabla\cdot\mathbf u^*.
$$

我们用周期域上的 MAC 交错网格：$u_{i+1/2,j}$ 位于竖直面，$v_{i,j+1/2}$ 位于水平面，$p_{i,j}$ 位于单元中心。本课用 $q=\Delta t\phi$ 作为投影势，因此实现 $\nabla_h^2q=\nabla_h\cdot\mathbf u^*$，然后减去 $\nabla_hq$。

**注意**：这仍是周期域，不包括实体壁面的边界条件。

## 2. 离散相容性：必须让散度与梯度能精确组合

$$
(D\mathbf u)_{i,j}=\frac{u_{i+1/2,j}-u_{i-1/2,j}}{\Delta x}
+\frac{v_{i,j+1/2}-v_{i,j-1/2}}{\Delta y}.
$$

$$
(G_xp)_{i+1/2,j}=\frac{p_{i+1,j}-p_{i,j}}{\Delta x},\qquad
(G_yp)_{i,j+1/2}=\frac{p_{i,j+1}-p_{i,j}}{\Delta y}.
$$

于是 $DG$ 就是五点差分拉普拉斯算子。以下代码使用 `jnp.roll`，**仅适用于周期边界**。

In [2]:
def grid(nx=64,ny=32,Lx=8.,Ly=4.):
 dx,dy=Lx/nx,Ly/ny
 x=jnp.arange(nx)*dx; y=jnp.arange(ny)*dy
 X,Y=jnp.meshgrid(x,y,indexing='ij')
 return dx,dy,X,Y

In [3]:
def div(u,v,dx,dy):
 return (u-jnp.roll(u,1,0))/dx+(v-jnp.roll(v,1,1))/dy

In [4]:
def grad(p,dx,dy):
 return (jnp.roll(p,-1,0)-p)/dx,(jnp.roll(p,-1,1)-p)/dy

In [5]:
def lap(q,dx,dy):
 return (jnp.roll(q,-1,0)+jnp.roll(q,1,0)-2*q)/(dx*dx)+(jnp.roll(q,-1,1)+jnp.roll(q,1,1)-2*q)/(dy*dy)

In [6]:
nx, ny, Lx, Ly = 48, 32, 3.0, 2.0
dx, dy, X, Y = grid(nx, ny, Lx, Ly)
p = jnp.sin(2*jnp.pi*X/Lx)*jnp.cos(2*jnp.pi*Y/Ly)
lhs = div(*grad(p,dx,dy),dx,dy)
rhs = lap(p,dx,dy)
compatibility_error = float(jnp.max(jnp.abs(lhs-rhs)))
print("|| D(G(p)) - Laplace(p) ||∞ =", compatibility_error)
assert compatibility_error < 1e-11

|| D(G(p)) - Laplace(p) ||∞ = 5.684341886080802e-14


## 3. FFT 解泊松方程

周期边界下，FFT 可以对角化离散拉普拉斯算子。其 Fourier 符号为：

$$
\lambda(k_x,k_y)=-\frac{4}{\Delta x^2}\sin^2\left(\frac{k_x\Delta x}{2}\right)
-\frac{4}{\Delta y^2}\sin^2\left(\frac{k_y\Delta y}{2}\right).
$$

零波数处 $\lambda=0$：压力只确定到一个常数。因此在数值实现中给零模选一个固定规范（平均压力为 0）。这也是**不能把周期 FFT 泊松求解器直接用于有入口、出口和固壁的流域**的原因。

In [7]:
def symbol(nx,ny,dx,dy):
 kx=2*jnp.pi*jnp.fft.fftfreq(nx,d=dx)[:,None]
 ky=2*jnp.pi*jnp.fft.fftfreq(ny,d=dy)[None,:]
 s=-4*jnp.sin(kx*dx/2)**2/dx**2-4*jnp.sin(ky*dy/2)**2/dy**2
 return jnp.where(s==0,1.,s)

In [10]:
def solve_poisson(rhs,s):
 rhs_hat=jnp.fft.fftn(rhs)
 p=jnp.fft.ifftn(rhs_hat/s).real
 return p-p.mean()

In [8]:
def project(u,v,dx,dy,s):
 p=solve_poisson(div(u,v,dx,dy),s)
 px,py=grad(p,dx,dy)
 return u-px,v-py,p

In [11]:
key = jax.random.PRNGKey(42)
ku, kv = jax.random.split(key)
u_star = jax.random.normal(ku, (nx,ny))
v_star = jax.random.normal(kv, (nx,ny))
S = symbol(nx,ny,dx,dy)
u,v,q = project(u_star,v_star,dx,dy,S)
before = np.asarray(div(u_star,v_star,dx,dy))
after = np.asarray(div(u,v,dx,dy))
print('投影前散度最大值:',float(np.max(np.abs(before))))
print('投影后散度最大值:',float(np.max(np.abs(after))))
assert np.max(np.abs(after)) < 1e-10

投影前散度最大值: 100.65295906376862
投影后散度最大值: 5.1514348342607263e-14


In [ ]:
fig,axs=plt.subplots(1,3,figsize=(12,3.3),constrained_layout=True)
for ax,a,title in zip(axs,[before,after,np.asarray(q)],['Before projection: div','After projection: div','Projection potential q']):
 im=ax.imshow(a.T,origin='lower',extent=(0,Lx,0,Ly),aspect='auto',cmap='RdBu_r')
 ax.set_title(title);fig.colorbar(im,ax=ax,shrink=.8)
plt.show()

## 4. 作业与验收

1. 修改 $N_x,N_y$：检查 $DG=\nabla_h^2$ 的相容误差是否仍然接近机器精度。
2. 将随机速度场改成 $u=\sin(2\pi x/L_x),v=0$，解释投影前后速度的差别。
3. 思考为什么压力泊松方程必须处理零频模式，以及将来非周期入口出口需要更换什么算法。

**本课通过条件**：散度最大误差 $<10^{-10}$；你能解释 FFT 为什么只用于这里的周期边界。